# v4: YOLO11m capacity test

Run with **Runtime → Run all**. The notebook trains `yolo11m` (COCO weights) on NEU-DET with the same recipe as the **v1r** control, so any gain comes from model size. Config: [`configs/train/v4_yolo11m_baseline_long.yaml`](../configs/train/v4_yolo11m_baseline_long.yaml)

## Checkpointing (nothing is lost on a disconnect)
- Training output (`runs/`) is a symlink to Google Drive, so every file Ultralytics writes goes straight to Drive.
- `last.pt` (weights, optimizer and epoch counter) is rewritten **after every epoch**, and `save_period: 1` also keeps an `epoch<N>.pt` snapshot of every epoch (about 40 MB each).
- If the session dies, **Run all** again: `scripts/train.py` finds `last.pt` and resumes from the last finished epoch.

## One-time setup
**Runtime → Change runtime type → T4 GPU** (or better). Make sure Google Drive has about 6 GB free per seed.

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/DEEPIKAGRACE22/PS-7---AI-Assisted-Quality-Inspection-IBM-x-MAH-Quantum-project-.git"
BRANCH = "feat/yolo11m-colab"   # change to "main" once this branch is merged
DRIVE_DIR = "/content/drive/MyDrive/ps7_yolo11m"   # checkpoints and results persist here

CONFIG = "configs/train/v4_yolo11m_baseline_long.yaml"
VERSION = "v4_yolo11m_baseline_long"
SEEDS = [0, 1, 2]          # use [0] for a quick first pass

In [ ]:
import os, subprocess, sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/ps7")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
# Keep training runs on Drive so checkpoints survive disconnects,
# and restore any model versions registered in earlier sessions.
drive_dir = Path(DRIVE_DIR)
(drive_dir / "runs").mkdir(parents=True, exist_ok=True)
(drive_dir / "models").mkdir(parents=True, exist_ok=True)

if not Path("runs").is_symlink():
    subprocess.run(["rm", "-rf", "runs"])
    Path("runs").symlink_to(drive_dir / "runs")

subprocess.run(["cp", "-rn", f"{drive_dir}/models/.", "models/"], check=True)

def run(*cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")

def save_to_drive():
    subprocess.run(["cp", "-r", "models/.", f"{drive_dir}/models/"], check=True)

def trained(version):
    return Path(f"models/{version}/weights/best.pt").exists()

def checkpoint_status(version):
    """Show the latest checkpoint on Drive for a run."""
    weights = Path("runs") / version / "weights"
    last = weights / "last.pt"
    if not last.exists():
        print(f"{version}: no checkpoint yet")
        return
    import time
    age = (time.time() - last.stat().st_mtime) / 60
    snapshots = len(list(weights.glob("epoch*.pt")))
    print(f"{version}: last.pt updated {age:.1f} min ago, {snapshots} epoch snapshots on Drive")

print("Registered versions:", sorted(p.name for p in Path("models").iterdir() if p.is_dir()))
for seed in SEEDS:
    checkpoint_status(f"{VERSION}_s{seed}")

## 1. Dataset

In [ ]:
run(sys.executable, "scripts/prepare_neu_det.py")

In [ ]:
from IPython.display import Image, display

run(sys.executable, "scripts/visualize_dataset.py", "data/neu_yolo", "--split", "train", "--n", "8")
display(Image("data/neu_yolo/preview_train.png"))

## 2. Training

One run per seed. If a seed already has `best.pt` in `models/`, it is skipped. If it was interrupted, it **resumes from `last.pt`** on Drive. Expect roughly 40–90 minutes per seed on a T4.

In [ ]:
for seed in SEEDS:
    version = f"{VERSION}_s{seed}"
    if trained(version):
        print(f"{version} already trained, skipping.")
        continue
    checkpoint_status(version)   # shows whether this run will resume
    run(sys.executable, "scripts/train.py", CONFIG, "--seed", seed, "--skip-existing")
    save_to_drive()              # copy the finished model (best.pt, config, metrics) to Drive
    checkpoint_status(version)

## 3. Evaluation

Validation is used to choose between models; test is reported once at the end. Metrics are written to `models/<version>/metrics/`.

In [ ]:
versions = sorted(p.parent.parent.name for p in Path("models").glob(f"{VERSION}_s*/weights/best.pt"))
for version in versions:
    for split in ("val", "test"):
        if not Path(f"models/{version}/metrics/{split}_metrics.csv").exists():
            run(sys.executable, "scripts/evaluate.py", version, "--split", split)

save_to_drive()

In [ ]:
import pandas as pd

rows = []
for version in versions:
    for split in ("val", "test"):
        row = pd.read_csv(f"models/{version}/metrics/{split}_metrics.csv").iloc[0]
        rows.append({"version": version, "split": split, "P": row.Precision, "R": row.Recall,
                     "mAP50": row.mAP50, "mAP50_95": row.mAP50_95})
summary = pd.DataFrame(rows)
display(summary.round(3))
display(summary.groupby("split")[["P", "R", "mAP50", "mAP50_95"]].agg(["mean", "std"]).round(3))

print("Per-class test recall:")
per_class = pd.concat(
    [pd.read_csv(f"models/{v}/metrics/test_per_class.csv").assign(version=v) for v in versions]
)
display(per_class.pivot(index="Class", columns="version", values="Recall").round(2))

## 4. Compare with the control

Compare against **v1r_yolo11n_baseline_long** (3 seeds): P 0.668, R 0.730, mAP50 0.737, mAP50-95 0.407 on test (see `models/comparison.csv`). Judge on validation, and look at crazing and rolled-in_scale recall separately.

## 5. Bring results back to the repo

Download the zip, unzip it into the repo's `models/` folder, add a row to `models/README.md`, and commit. Each `best.pt` is about 40 MB.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/ps7_models", "zip", "models")
print("Also saved on Drive:", drive_dir / "models")
files.download(archive)